In [ ]:
import pandas as pd
import numpy as np
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import importlib
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df  = pd.read_csv(data_folder + 'train.csv')
raw_test_df   = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'

model_run_name = 'lgbm.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits   = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013','SEQ_000034', 'SEQ_000046', 'SEQ_000053']

model_target = 'gesture_action'

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target']

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(target_only_df, train_pct=0.2, test_pct=0.2)

# train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_sample_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain':             ['acceleration'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [['rot_w', 'rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes':       [True],
    f'{pipe_name}__rotation_domain':        ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1']],

    f'{pipe_name}__tof_sensor_list':        [tof_1],
    f'{pipe_name}__tof_mode':               ['research'],

    f'{pipe_name}__window':                 [0.2],
    f'{pipe_name}__step_sec':               [0.037],
    f'{pipe_name}__dc_offset':              [1.5],
    f'{pipe_name}__band_edges':             [None],
    f'{pipe_name}__category_data':          [True],
    f'{pipe_name}__segmentation':           ['window'],

    'classifier__estimator__n_estimators':    [300],
    'classifier__estimator__max_depth':       [4],
    'classifier__estimator__learning_rate':   [0.05],
    'classifier__estimator__subsample':       [0.8],
    'classifier__estimator__colsample_bytree':[0.7],
    'classifier__estimator__min_child_samples':[50],
    'classifier__estimator__reg_alpha':       [1],
    'classifier__estimator__reg_lambda':      [10],
    'classifier__estimator__num_leaves':      [8],
    'classifier__estimator__min_gain_to_split':[1.0],  # add this
}

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

lgbm_clf = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=lgbm_clf,
        extractor=custom_extractor,
        mode=None,
        target = model_target
    ))
])

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', model_target]]

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])
pbar.close()

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'lgbm_results.csv', index=False)

In [ ]:
cv_df = cv_results_df.sort_values('rank_test_score').T.reset_index()

split_frame_series = cv_df['index'].str.replace('__', '_').str.split('_')
imu_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'imu')
classifier_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'classifier')

cv_df.loc[split_frame_series.str[-1] == 'time', 'name'] = split_frame_series.str[0:2].str.join('_')
cv_df.loc[split_frame_series.str[-1] == 'time', 'eval'] = 'time'

cv_df.loc[classifier_rows, 'eval'] = 'classifier'
cv_df.loc[classifier_rows, 'name'] = split_frame_series.loc[classifier_rows].str[3:].str.join('_')

cv_df.loc[imu_rows, 'eval'] = 'imu'
cv_df.loc[imu_rows, 'name'] = split_frame_series.loc[imu_rows].str[3:].str.join('_')

cv_df.loc[split_frame_series.str[0] == 'params', 'eval'] = 'params'
cv_df.loc[split_frame_series.str[0] == 'params', 'name'] = 'params'

cv_df.loc[cv_df['index'].str.startswith('split'), 'eval'] = 'split_score'
cv_df.loc[cv_df['index'].str.startswith('split'), 'name'] = split_frame_series.str[0]

cv_df.loc[cv_df['index'] == 'mean_train_score', 'eval'] = 'train_score'
cv_df.loc[cv_df['index'] == 'mean_train_score', 'name'] = 'mean'

cv_df.loc[cv_df['index'] == 'mean_test_score', 'eval'] = 'test_score'
cv_df.loc[cv_df['index'] == 'mean_test_score', 'name'] = 'mean'

cv_df.loc[cv_df['index'] == 'std_train_score', 'eval'] = 'train_score'
cv_df.loc[cv_df['index'] == 'std_train_score', 'name'] = 'std'

cv_df.loc[cv_df['index'] == 'std_test_score', 'eval'] = 'test_score'
cv_df.loc[cv_df['index'] == 'std_test_score', 'name'] = 'std'

cv_df.loc[cv_df['index'] == 'rank_test_score', ['eval', 'name']] = 'rank'
cv_df.drop(columns='index', inplace=True)

to_view = cv_df[cv_df['eval'] != 'classifier']
to_view = to_view[to_view['eval'] != 'time']
to_view = to_view[to_view['name'] != 'std'].set_index(['eval', 'name']).T
to_view

In [ ]:
best_model = grid_search.best_estimator_

extractor    = best_model.named_steps['imu_extractor']
preprocessor = best_model.named_steps['preprocessor']
classifier   = best_model.named_steps['classifier']

X_feat = extractor.transform(test_sample_df)
X_proc = preprocessor.transform(X_feat)

y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
y_true = y_true.reindex(X_proc.index)

y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

print(classification_report(y_true, y_pred))

report_df = pd.DataFrame(
    classification_report(y_true, y_pred, output_dict=True)
).T.sort_values('f1-score', ascending=False)

report_df.to_csv(model_run_folder_name + 'per_gesture_scores.csv')